# EGM vs NEGM: selfgen parameter recovery

Compare how EGM(FUES) and NEGM(FUES) recover the true calibration parameters from selfgen data. Each method solves the model at uniform CE draws, simulates, and matches moments generated from the YAML defaults. The true parameters are known — this is a controlled recovery test.

For each run we report: estimated theta, distance from truth, SE, convergence, and runtime.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*IProgress.*')

import json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath('../../..')
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ── Gadi mount ──
GADI = os.path.expanduser('~/gadi/g/data/tp66/results/durables2_0/estimation')

def list_runs(spec_prefix=None):
    """List all available runs, optionally filtered by spec prefix."""
    runs = []
    for spec in sorted(os.listdir(GADI)):
        spec_dir = os.path.join(GADI, spec)
        if not os.path.isdir(spec_dir):
            continue
        if spec_prefix and not spec.startswith(spec_prefix):
            continue
        for item in sorted(os.listdir(spec_dir)):
            p = os.path.join(spec_dir, item)
            if os.path.isfile(os.path.join(p, 'summary.json')):
                runs.append(f'{spec}/{item}')
            elif os.path.isdir(p):
                for sub in sorted(os.listdir(p)):
                    if os.path.isfile(os.path.join(p, sub, 'summary.json')):
                        runs.append(f'{spec}/{item}/{sub}')
    return runs

def load_run(run_path):
    """Load summary.json from a run path (relative to GADI)."""
    full = os.path.join(GADI, run_path, 'summary.json')
    with open(full) as f:
        return json.load(f)

# Show available selfgen runs
print('Available selfgen runs:')
for r in list_runs('selfgen'):
    print(f'  {r}')

## 1. Select runs

Pick one EGM and one NEGM selfgen run to compare. Use the latest converged run for each method.

In [ ]:
# ── Select runs to compare ──
# Change these paths to point at different runs.
RUNS = {
    'EGM':  'selfgen_large_egm/est_20260328_012600',
    'NEGM': 'selfgen_large_negm/est_20260328_012600',
}

# True parameters (YAML calibration defaults for separable model)
THETA_TRUE = {
    'beta':    0.945,
    'alpha':   0.7,
    'gamma_c': 3.5,
    'gamma_h': 1.5,
    'tau':     0.12,
}

# Load
results = {}
for label, path in RUNS.items():
    results[label] = load_run(path)
    s = results[label]
    print(f'{label}: loss={s["objective"]:.2e}, converged={s["converged"]}, '
          f'n_iter={s["n_iter"]}')

## 2. Parameter recovery table

In [ ]:
params = sorted(THETA_TRUE.keys())

rows = []
for p in params:
    row = {'Parameter': p, 'True': THETA_TRUE[p]}
    for label in RUNS:
        s = results[label]
        est = s['theta_best'][p]
        se = s['theta_se'].get(p, float('nan'))
        err = est - THETA_TRUE[p]
        row[f'{label} est'] = est
        row[f'{label} SE'] = se
        row[f'{label} err'] = err
    rows.append(row)

# Add summary row
row_loss = {'Parameter': 'Loss'}
row_iter = {'Parameter': 'Iterations'}
row_conv = {'Parameter': 'Converged'}
for label in RUNS:
    s = results[label]
    row_loss[f'{label} est'] = s['objective']
    row_iter[f'{label} est'] = s['n_iter']
    row_conv[f'{label} est'] = s['converged']
    for col in [f'{label} SE', f'{label} err', 'True']:
        row_loss[col] = ''
        row_iter[col] = ''
        row_conv[col] = ''

df = pd.DataFrame(rows + [row_loss, row_iter, row_conv])

# Display
from IPython.display import Markdown

def _fmt(v):
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        if abs(v) < 1e-3 and v != 0:
            return f'{v:.2e}'
        return f'{v:.6f}'
    if isinstance(v, int):
        return str(v)
    return str(v)

header = '| Parameter | True |'
sep = '|---|---|'
for label in RUNS:
    header += f' {label} est | {label} SE | {label} err |'
    sep += '---|---|---|'

lines = [header, sep]
for _, r in df.iterrows():
    line = f'| {r["Parameter"]} | {_fmt(r["True"])} |'
    for label in RUNS:
        line += f' {_fmt(r[f"{label} est"])} | {_fmt(r[f"{label} SE"])} | {_fmt(r[f"{label} err"])} |'
    lines.append(line)

Markdown('\n'.join(lines))

## 3. Recovery accuracy: bar chart

In [ ]:
# Percentage error |theta_best - theta_true| / theta_true
labels_method = list(RUNS.keys())
x = np.arange(len(params))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
for i, label in enumerate(labels_method):
    s = results[label]
    pct_err = [100 * abs(s['theta_best'][p] - THETA_TRUE[p]) / THETA_TRUE[p]
               for p in params]
    ax.bar(x + i * width, pct_err, width, label=label)

ax.set_ylabel('|error| / true (%)')
ax.set_title('Parameter recovery: percentage error')
ax.set_xticks(x + width / 2)
ax.set_xticklabels([f'${p}$' for p in params])
ax.legend()
ax.set_yscale('log')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Convergence comparison

In [ ]:
# Load convergence CSVs and plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, (label, path) in enumerate(RUNS.items()):
    conv_path = os.path.join(GADI, path, 'convergence.csv')
    if not os.path.exists(conv_path):
        print(f'{label}: no convergence.csv')
        continue
    conv = pd.read_csv(conv_path)

    ax = axes[0]
    ax.plot(conv['iter'], conv['best_loss'], 'o-', label=label, markersize=3)

    ax = axes[1]
    ax.plot(conv['iter'], conv['elite_mean_loss'], 's-', label=label, markersize=3)

axes[0].set_xlabel('CE iteration')
axes[0].set_ylabel('Best loss')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].set_title('Best loss')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('CE iteration')
axes[1].set_ylabel('Elite mean loss')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].set_title('Elite mean loss')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Convergence: EGM vs NEGM', fontsize=13)
fig.tight_layout()
plt.show()

## 5. Sweep comparison: recovery across gamma_c values

If sweep results are available, compare how each method recovers the remaining 4 parameters at each fixed gamma_c.

In [ ]:
# ── Select sweep runs ──
# Set to None to skip this section.
SWEEP_EGM_SPEC = 'selfgen_sweep_gamma_c_egm'
SWEEP_NEGM_SPEC = 'selfgen_sweep_gamma_c_negm'

# True values for the 4 free params in gamma_c sweeps (gamma_c is fixed)
SWEEP_TRUE = {
    'beta': 0.945, 'alpha': 0.7, 'gamma_h': 1.5, 'tau': 0.12,
}

def _load_sweep(spec_name):
    """Load all sweep points for a spec, returning {gamma_c_val: summary}."""
    out = {}
    spec_dir = os.path.join(GADI, spec_name)
    if not os.path.isdir(spec_dir):
        return out
    for point_dir in sorted(os.listdir(spec_dir)):
        point_path = os.path.join(spec_dir, point_dir)
        if not os.path.isdir(point_path):
            continue
        # Find the latest run in this point
        run_dirs = sorted([d for d in os.listdir(point_path)
                           if os.path.isfile(os.path.join(point_path, d, 'summary.json'))])
        if not run_dirs:
            continue
        latest = run_dirs[-1]
        with open(os.path.join(point_path, latest, 'summary.json')) as f:
            s = json.load(f)
        # Extract gamma_c value from dir name
        if '=' in point_dir:
            val = float(point_dir.split('=')[1])
            out[val] = s
    return out

sweep_egm = _load_sweep(SWEEP_EGM_SPEC) if SWEEP_EGM_SPEC else {}
sweep_negm = _load_sweep(SWEEP_NEGM_SPEC) if SWEEP_NEGM_SPEC else {}

print(f'EGM sweep:  {len(sweep_egm)} points')
print(f'NEGM sweep: {len(sweep_negm)} points')

if sweep_egm or sweep_negm:
    sweep_params = sorted(SWEEP_TRUE.keys())
    gamma_c_vals = sorted(set(list(sweep_egm.keys()) + list(sweep_negm.keys())))

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()

    for i, p in enumerate(sweep_params):
        ax = axes[i]
        true_val = SWEEP_TRUE[p]
        ax.axhline(true_val, color='k', linestyle=':', linewidth=1, label='True')

        if sweep_egm:
            gc_vals = sorted(sweep_egm.keys())
            est = [sweep_egm[g]['theta_best'].get(p, np.nan) for g in gc_vals]
            ax.plot(gc_vals, est, 'bo-', label='EGM', markersize=5)

        if sweep_negm:
            gc_vals = sorted(sweep_negm.keys())
            est = [sweep_negm[g]['theta_best'].get(p, np.nan) for g in gc_vals]
            ax.plot(gc_vals, est, 'rs--', label='NEGM', markersize=5)

        ax.set_xlabel(r'Fixed $\gamma_c$')
        ax.set_ylabel(f'Estimated ${p}$')
        ax.set_title(f'${p}$ (true = {true_val})')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(r'Parameter recovery across fixed $\gamma_c$', fontsize=13)
    fig.tight_layout()
    plt.show()

    # Loss comparison
    fig, ax = plt.subplots(figsize=(8, 4))
    if sweep_egm:
        gc_vals = sorted(sweep_egm.keys())
        losses = [sweep_egm[g]['objective'] for g in gc_vals]
        ax.plot(gc_vals, losses, 'bo-', label='EGM', markersize=5)
    if sweep_negm:
        gc_vals = sorted(sweep_negm.keys())
        losses = [sweep_negm[g]['objective'] for g in gc_vals]
        ax.plot(gc_vals, losses, 'rs--', label='NEGM', markersize=5)
    ax.set_xlabel(r'Fixed $\gamma_c$')
    ax.set_ylabel('SMM loss')
    ax.set_yscale('log')
    ax.set_title(r'Loss across fixed $\gamma_c$')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print('No sweep data available.')

## 6. All selfgen runs summary

Load all available selfgen runs and tabulate for quick comparison.

In [ ]:
# Load all selfgen runs
all_runs = list_runs('selfgen')
rows = []
for path in all_runs:
    try:
        s = load_run(path)
    except Exception:
        continue
    # Determine method from spec name
    method = 'NEGM' if 'negm' in path.lower() else 'EGM'
    # Determine if sweep and which param
    sweep_param = ''
    if 'sweep_gamma_c' in path:
        parts = path.split('/')
        for p in parts:
            if p.startswith('gamma_c='):
                sweep_param = p
    elif 'sweep_sigma_w' in path:
        parts = path.split('/')
        for p in parts:
            if p.startswith('sigma_w='):
                sweep_param = p

    row = {
        'run': path.split('/')[-1],  # est_YYYYMMDD_HHMMSS
        'spec': path.split('/')[0],
        'method': method,
        'sweep': sweep_param,
        'loss': s['objective'],
        'converged': s['converged'],
        'n_iter': s['n_iter'],
    }
    for p in sorted(THETA_TRUE.keys()):
        if p in s['theta_best']:
            row[p] = s['theta_best'][p]
    rows.append(row)

all_df = pd.DataFrame(rows)
# Sort by spec then run
all_df = all_df.sort_values(['spec', 'sweep', 'run'])

# Filter to converged runs with non-trivial loss
valid = all_df[(all_df['converged'] == True) & (all_df['loss'] > 1e-20)]
print(f'{len(valid)} converged runs with non-trivial loss (out of {len(all_df)} total)')
print()

cols = ['spec', 'method', 'sweep', 'loss', 'n_iter'] + sorted(THETA_TRUE.keys())
print(valid[cols].to_string(index=False, float_format='%.6f'))

---

*Source: `examples/durables2_0/` — Dobrescu and Shanker (2026), Application 2*